# 🧪 MongoDB NoSQL Database Systems — Interactive Step-by-Step Hands-On Lab

Welcome to the **MongoDB Interactive Lab Notebook**! This hands-on lab guides you step-by-step through every core MongoDB topic, from database and collection management to CRUD operations, cursor modifiers (`sort`, `skip`, `limit`), projections, and advanced query/update operators.

### 🗺️ Lab Curriculum Overview
| Module | Topic | Commands & Operators Covered |
| :--- | :--- | :--- |
| **Lab 0** | **Environment Setup** | Initializing the `%%mongosh` Shell Engine & Automated Lab Grader |
| **Lab 1** | **Database Operations** | `use <db>`, lazy database creation on first insert, `show dbs`, `db.dropDatabase()` |
| **Lab 2** | **Collection Operations** | `db.createCollection()`, `show collections`, `db.<collection>.drop()` |
| **Lab 3** | **CRUD: Create & Read** | `insertOne()`, `insertMany()`, `find()`, `findOne()` |
| **Lab 4** | **CRUD: Update & Delete** | `updateOne()`, `updateMany()`, `deleteOne()`, `deleteMany()` |
| **Lab 5** | **More About `find()`** | Query Filters, Projections (`1`/`0`), `sort()`, `skip()`, `limit()`, `findOne()` Revisited |
| **Lab 6** | **Query & Update Operators** | Comparison (`$gt`, `$gte`, `$lt`, `$lte`, `$ne`, `$in`, `$nin`), Logical (`$or`, `$and`), Update (`$set`, `$inc`, `$push`, `$pull`) |
| **Lab 7** | **Capstone Lab Challenge** | End-to-End Multi-Step Database Workflow & Final Scoreboard |

---
### ⚙️ How to Use This Lab Notebook
1. **Run the Setup Cell (Lab 0)** first by pressing **`Shift + Enter`**.
2. Read each **Concept Walkthrough**, run the **Guided Demo** cell, and then complete the **Hands-On Lab Task** cell using standard MongoDB Shell (`mongosh`) syntax starting with `%%mongosh`.
3. Run the **Automated Verification Cell** right below each task to check your work and record your score!

---
## 🛠️ Lab 0: Initialize the MongoDB Shell (`%%mongosh`) & Auto-Grader Engine
Run the code cell below once to start the built-in interactive MongoDB Shell environment and task verification system. It works out-of-the-box in any Jupyter environment without requiring external server configuration.

In [1]:
# ============================================================================
# 🛠️ MONGODB INTERACTIVE LAB ENGINE & AUTO-GRADER SETUP
# Run this cell first (Shift + Enter) to initialize the MongoDB environment!
# ============================================================================
import copy
import json
import re

class MongoCursor:
    def __init__(self, docs):
        self._docs = [copy.deepcopy(d) for d in docs]
        self._sort_spec = None
        self._skip_n = 0
        self._limit_n = None

    def sort(self, spec):
        self._sort_spec = spec
        return self

    def skip(self, n):
        self._skip_n = int(n)
        return self

    def limit(self, n):
        self._limit_n = int(n)
        return self

    def to_list(self):
        res = [copy.deepcopy(d) for d in self._docs]
        if self._sort_spec:
            for field, direction in reversed(list(self._sort_spec.items())):
                res.sort(key=lambda x: (x.get(field) is None, x.get(field)), reverse=(direction == -1))
        if self._skip_n:
            res = res[self._skip_n:]
        if self._limit_n is not None:
            res = res[:self._limit_n]
        return res

    def __iter__(self):
        return iter(self.to_list())

    def __repr__(self):
        return json.dumps(self.to_list(), indent=2)


def _match_condition(val, cond):
    if isinstance(cond, dict) and any(k.startswith("$") for k in cond.keys()):
        for op, target in cond.items():
            if op == "$gt":
                if val is None or not (val > target): return False
            elif op == "$gte":
                if val is None or not (val >= target): return False
            elif op == "$lt":
                if val is None or not (val < target): return False
            elif op == "$lte":
                if val is None or not (val <= target): return False
            elif op == "$ne":
                if val == target: return False
            elif op == "$in":
                if isinstance(val, list):
                    if not any(v in target for v in val): return False
                else:
                    if val not in target: return False
            elif op == "$nin":
                if isinstance(val, list):
                    if any(v in target for v in val): return False
                else:
                    if val in target: return False
            else:
                raise ValueError(f"Unsupported operator: {op}")
        return True
    else:
        if isinstance(val, list) and not isinstance(cond, list):
            return cond in val
        return val == cond


def _match_filter(doc, query):
    if not query:
        return True
    for k, v in query.items():
        if k == "$or":
            if not any(_match_filter(doc, sub) for sub in v):
                return False
        elif k == "$and":
            if not all(_match_filter(doc, sub) for sub in v):
                return False
        else:
            if not _match_condition(doc.get(k), v):
                return False
    return True


def _apply_projection(doc, proj):
    if not proj:
        return copy.deepcopy(doc)
    inc_keys = [k for k, v in proj.items() if k != "_id" and v == 1]
    exc_keys = [k for k, v in proj.items() if k != "_id" and v == 0]
    include_id = proj.get("_id", 1) != 0

    if inc_keys or (len(proj) == 1 and "_id" in proj and not include_id):
        if inc_keys:
            out = {}
            if include_id and "_id" in doc:
                out["_id"] = doc["_id"]
            for k in inc_keys:
                if k in doc:
                    out[k] = copy.deepcopy(doc[k])
            return out
        else:
            out = copy.deepcopy(doc)
            out.pop("_id", None)
            return out
    else:
        out = copy.deepcopy(doc)
        if not include_id:
            out.pop("_id", None)
        for k in exc_keys:
            out.pop(k, None)
        return out


def _apply_update(doc, update):
    for op, fields in update.items():
        if op == "$set":
            for k, v in fields.items():
                doc[k] = copy.deepcopy(v)
        elif op == "$inc":
            for k, v in fields.items():
                doc[k] = doc.get(k, 0) + v
        elif op == "$push":
            for k, v in fields.items():
                if k not in doc or not isinstance(doc[k], list):
                    doc[k] = []
                doc[k].append(copy.deepcopy(v))
        elif op == "$pull":
            for k, v in fields.items():
                if k in doc and isinstance(doc[k], list):
                    doc[k] = [item for item in doc[k] if item != v]
        else:
            raise ValueError(f"Unsupported update operator: {op}")


class MongoCollection:
    def __init__(self, db_ref, name):
        self._db = db_ref
        self._name = name
        self._docs = []
        self._counter = 1

    def _ensure_created(self):
        self._db._register_collection(self._name, self)

    def insertOne(self, doc):
        self._ensure_created()
        new_doc = copy.deepcopy(doc)
        if "_id" not in new_doc:
            new_doc["_id"] = f"ObjectId('{self._counter:024x}')"
            self._counter += 1
        self._docs.append(new_doc)
        return {"acknowledged": True, "insertedId": new_doc["_id"]}

    def insertMany(self, docs):
        self._ensure_created()
        ids = {}
        for idx, d in enumerate(docs):
            res = self.insertOne(d)
            ids[str(idx)] = res["insertedId"]
        return {"acknowledged": True, "insertedIds": ids}

    def find(self, query=None, projection=None):
        matched = [_apply_projection(d, projection) for d in self._docs if _match_filter(d, query or {})]
        return MongoCursor(matched)

    def findOne(self, query=None, projection=None):
        for d in self._docs:
            if _match_filter(d, query or {}):
                return _apply_projection(d, projection)
        return None

    def updateOne(self, query, update):
        matched = 0
        modified = 0
        for d in self._docs:
            if _match_filter(d, query):
                matched = 1
                before = copy.deepcopy(d)
                _apply_update(d, update)
                if d != before:
                    modified = 1
                break
        return {"acknowledged": True, "matchedCount": matched, "modifiedCount": modified}

    def updateMany(self, query, update):
        matched = 0
        modified = 0
        for d in self._docs:
            if _match_filter(d, query):
                matched += 1
                before = copy.deepcopy(d)
                _apply_update(d, update)
                if d != before:
                    modified += 1
        return {"acknowledged": True, "matchedCount": matched, "modifiedCount": modified}

    def deleteOne(self, query):
        for i, d in enumerate(self._docs):
            if _match_filter(d, query):
                self._docs.pop(i)
                return {"acknowledged": True, "deletedCount": 1}
        return {"acknowledged": True, "deletedCount": 0}

    def deleteMany(self, query):
        initial_len = len(self._docs)
        self._docs = [d for d in self._docs if not _match_filter(d, query)]
        deleted = initial_len - len(self._docs)
        return {"acknowledged": True, "deletedCount": deleted}

    def drop(self):
        return self._db._drop_collection(self._name)


class MongoDatabase:
    def __init__(self, server, name):
        self._server = server
        self._name = name
        self._collections = {}

    def _register_collection(self, name, col):
        self._collections[name] = col
        self._server._databases[self._name] = self

    def _drop_collection(self, name):
        if name in self._collections:
            del self._collections[name]
            if not self._collections:
                self._server._databases.pop(self._name, None)
            return True
        return False

    def createCollection(self, name, options=None):
        if name not in self._collections:
            self._collections[name] = MongoCollection(self, name)
        self._server._databases[self._name] = self
        return {"ok": 1}

    def dropDatabase(self):
        self._collections.clear()
        self._server._databases.pop(self._name, None)
        return {"ok": 1, "dropped": self._name}

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        if name not in self._collections:
            return MongoCollection(self, name)
        return self._collections[name]


class MongoServer:
    def __init__(self):
        self._databases = {}
        self.current_db_name = "test"
        self._pending_dbs = {"test": MongoDatabase(self, "test")}
        self.history = []
        self.last_output = None
        self.last_batch_outputs = []
        self.scores = {}

    @property
    def current_db(self):
        if self.current_db_name in self._databases:
            return self._databases[self.current_db_name]
        if self.current_db_name not in self._pending_dbs:
            self._pending_dbs[self.current_db_name] = MongoDatabase(self, self.current_db_name)
        return self._pending_dbs[self.current_db_name]

    def use_db(self, db_name):
        self.current_db_name = db_name
        if db_name not in self._databases and db_name not in self._pending_dbs:
            self._pending_dbs[db_name] = MongoDatabase(self, db_name)
        return f"switched to db {db_name}"

    def show_dbs(self):
        lines = [
            "admin      0.000GB",
            "config     0.000GB",
            "local      0.000GB"
        ]
        for db_name, db_obj in sorted(self._databases.items()):
            if len(db_obj._collections) > 0 and db_name not in ("admin", "config", "local"):
                lines.append(f"{db_name:<10} 0.000GB")
        return "\n".join(lines)

    def show_collections(self):
        db_obj = self._databases.get(self.current_db_name)
        if not db_obj or not db_obj._collections:
            return "(empty - no collections exist yet)"
        return "\n".join(sorted(db_obj._collections.keys()))


_MONGO_SERVER = MongoServer()

class DBProxy:
    def __getattr__(self, name):
        return getattr(_MONGO_SERVER.current_db, name)

db = DBProxy()

def _js_to_py_expr(stmt):
    strings = []
    def save_str(m):
        strings.append(m.group(0))
        return f"__STR_PLACEHOLDER_{len(strings)-1}__"

    s = re.sub(r'"(?:\\.|[^"\\])*"|\'(?:\\.|[^\'\\])*\'', save_str, stmt)
    s = re.sub(r'//.*', '', s)
    s = re.sub(r'\btrue\b', 'True', s)
    s = re.sub(r'\bfalse\b', 'False', s)
    s = re.sub(r'\bnull\b', 'None', s)
    s = re.sub(r'([\{,]\s*)([\$a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1"\2":', s)
    for idx, orig in enumerate(strings):
        s = s.replace(f"__STR_PLACEHOLDER_{idx}__", orig)
    return s.strip()

def run_mongosh(script_text):
    """Executes mongosh commands step-by-step and prints realistic shell output."""
    _MONGO_SERVER.history.append(script_text)
    cleaned_lines = []
    for raw_line in script_text.splitlines():
        line_no_comment = re.sub(r'(?<!:)//.*$', '', raw_line).strip()
        if not line_no_comment:
            continue
        cleaned_lines.append(line_no_comment)

    statements = []
    buf = []
    depth = 0
    for line in cleaned_lines:
        if depth == 0 and re.match(r'^(use\s+\w+|show\s+dbs|show\s+collections)\s*;?$', line):
            statements.append(line.rstrip(';').strip())
            continue
        for ch in line:
            if ch in '([{':
                depth += 1
            elif ch in ')]}':
                depth = max(0, depth - 1)
        buf.append(line)
        if depth == 0:
            joined = " ".join(buf).strip()
            sub_buf = []
            sub_depth = 0
            in_str = None
            for ch in joined:
                if in_str:
                    sub_buf.append(ch)
                    if ch == in_str:
                        in_str = None
                else:
                    if ch in ('"', "'"):
                        in_str = ch
                        sub_buf.append(ch)
                    elif ch in '([{':
                        sub_depth += 1
                        sub_buf.append(ch)
                    elif ch in ')]}':
                        sub_depth = max(0, sub_depth - 1)
                        sub_buf.append(ch)
                    elif ch == ';' and sub_depth == 0:
                        st = "".join(sub_buf).strip()
                        if st:
                            statements.append(st)
                        sub_buf = []
                    else:
                        sub_buf.append(ch)
            rem = "".join(sub_buf).strip()
            if rem:
                statements.append(rem)
            buf = []

    outputs = []
    for st in statements:
        m_use = re.match(r'^use\s+([a-zA-Z0-9_]+)$', st)
        if m_use:
            out = _MONGO_SERVER.use_db(m_use.group(1))
            print(f"{_MONGO_SERVER.current_db_name}> {st}\n{out}\n")
            outputs.append(out)
            continue
        if st == "show dbs":
            out = _MONGO_SERVER.show_dbs()
            print(f"{_MONGO_SERVER.current_db_name}> {st}\n{out}\n")
            outputs.append(out)
            continue
        if st == "show collections":
            out = _MONGO_SERVER.show_collections()
            print(f"{_MONGO_SERVER.current_db_name}> {st}\n{out}\n")
            outputs.append(out)
            continue

        py_expr = _js_to_py_expr(st)
        if not py_expr:
            continue
        res = eval(py_expr, {"db": db, "True": True, "False": False, "None": None})
        if isinstance(res, MongoCursor):
            val = res.to_list()
        else:
            val = res
        _MONGO_SERVER.last_output = val
        outputs.append(val)
        formatted = json.dumps(val, indent=2) if val is not None else "null"
        print(f"{_MONGO_SERVER.current_db_name}> {st}\n{formatted}\n")
    _MONGO_SERVER.last_batch_outputs = outputs
    return outputs

try:
    from IPython.core.magic import register_cell_magic
    @register_cell_magic
    def mongosh(line, cell):
        run_mongosh(cell)
except Exception:
    pass

def record_grade(task_id, passed, msg_pass, msg_fail):
    _MONGO_SERVER.scores[task_id] = passed
    if passed:
        print(f"✅ [PASS] {task_id}: {msg_pass}")
    else:
        print(f"❌ [FAIL] {task_id}: {msg_fail}")

def show_lab_report():
    total = len(_MONGO_SERVER.scores)
    passed = sum(1 for v in _MONGO_SERVER.scores.values() if v)
    print("=" * 65)
    print(f"🏆 MONGODB HANDS-ON LAB SCOREBOARD: {passed}/{total} Tasks Completed")
    print("=" * 65)
    for tid, status in _MONGO_SERVER.scores.items():
        icon = "✅ PASS" if status else "❌ PENDING"
        print(f"  {icon}  |  {tid}")
    print("=" * 65)

print("✅ MongoDB Interactive Lab Engine & Grader Ready!")
print("👉 Prefix MongoDB Shell cells with %%mongosh and press Shift+Enter to execute.")

✅ MongoDB Interactive Lab Engine & Grader Ready!
👉 Prefix MongoDB Shell cells with %%mongosh and press Shift+Enter to execute.


---
## 📂 Lab 1: Database Operations (`use`, `show dbs`, `db.dropDatabase()`)

### 📖 Concept Walkthrough
1. **Create & Switch Database (`use <databaseName>`):**
   - Switches the current database context for all subsequent commands in the session.
   - MongoDB has **no dedicated command** to create a database ahead of time. Switching to a non-existent database selects it in memory, but it is **only created on disk** once you insert data into a collection within it.
2. **List Databases (`show dbs`):**
   - Lists every database on the connected MongoDB server along with its storage size.
   - A database only appears in `show dbs` after data has been written to it.
3. **Delete Database (`db.dropDatabase()`):**
   - Permanently deletes the currently selected database, including all of its collections and documents. This action cannot be undone.

### 🔍 Guided Demo 1: See How Lazy Database Creation Works
Run the demo cell below to observe how `use demoDB` alone does not list `demoDB` in `show dbs` until a document is inserted, and how `db.dropDatabase()` removes it.

In [2]:
%%mongosh
// 1. Switch to demoDB (selected, but not yet on disk)
use demoDB
show dbs

// 2. Insert a document so demoDB is created on disk
db.items.insertOne({ item: "Notebook", qty: 5 });
show dbs

// 3. Clean up demoDB using dropDatabase()
db.dropDatabase();
show dbs

demoDB> use demoDB
switched to db demoDB

demoDB> show dbs
admin      0.000GB
config     0.000GB
local      0.000GB

demoDB> db.items.insertOne({ item: "Notebook", qty: 5 })
{
  "acknowledged": true,
  "insertedId": "ObjectId('000000000000000000000001')"
}

demoDB> show dbs
admin      0.000GB
config     0.000GB
local      0.000GB
demoDB     0.000GB

demoDB> db.dropDatabase()
{
  "ok": 1,
  "dropped": "demoDB"
}

demoDB> show dbs
admin      0.000GB
config     0.000GB
local      0.000GB



### 🎯 Hands-On Task 1.1: Select & Materialize `schoolDB`
**Assigned Task:**
1. Switch the current database context to `schoolDB` using `use`.
2. Insert a single document `{ name: "Aditi", age: 18, grade: 8 }` into the `students` collection using `db.students.insertOne(...)` so `schoolDB` is created on disk.
3. Run `show dbs` to verify that `schoolDB` now appears in the server's database list.

In [3]:
%%mongosh
// Step 1: Switch to schoolDB
use schoolDB

// Step 2: Insert Aditi's document into the students collection
db.students.insertOne({ name: "Aditi", age: 18, grade: 8 });

// Step 3: List all databases to confirm schoolDB exists
show dbs

schoolDB> use schoolDB
switched to db schoolDB

schoolDB> db.students.insertOne({ name: "Aditi", age: 18, grade: 8 })
{
  "acknowledged": true,
  "insertedId": "ObjectId('000000000000000000000001')"
}

schoolDB> show dbs
admin      0.000GB
config     0.000GB
local      0.000GB
schoolDB   0.000GB



In [4]:
# 🧪 Automated Verification — Task 1.1
has_school_db = "schoolDB" in _MONGO_SERVER._databases
aditi_doc = _MONGO_SERVER._databases.get("schoolDB").students.findOne({"name": "Aditi"}) if has_school_db else None
record_grade(
    "Task 1.1: Create & Use schoolDB",
    bool(has_school_db and aditi_doc and aditi_doc.get("age") == 18),
    "schoolDB is active on disk and contains Aditi's student record!",
    "Make sure you switched to schoolDB and inserted { name: 'Aditi', age: 18, grade: 8 } into db.students."
)

✅ [PASS] Task 1.1: Create & Use schoolDB: schoolDB is active on disk and contains Aditi's student record!


### 🎯 Hands-On Task 1.2: Create and Drop a Temporary Database (`sandboxDB`)
**Assigned Task:**
1. Switch to a temporary database named `sandboxDB`.
2. Insert `{ temp: true }` into `db.logs` so `sandboxDB` is created.
3. Permanently delete `sandboxDB` using `db.dropDatabase()`.
4. Switch back to `schoolDB` and run `show dbs` to confirm `sandboxDB` is gone while `schoolDB` remains intact.

In [5]:
%%mongosh
// Step 1: Switch to sandboxDB and insert a dummy document
use sandboxDB
db.logs.insertOne({ temp: true });

// Step 2: Permanently drop sandboxDB
db.dropDatabase();

// Step 3: Switch back to schoolDB and verify with show dbs
use schoolDB
show dbs

sandboxDB> use sandboxDB
switched to db sandboxDB

sandboxDB> db.logs.insertOne({ temp: true })
{
  "acknowledged": true,
  "insertedId": "ObjectId('000000000000000000000001')"
}

sandboxDB> db.dropDatabase()
{
  "ok": 1,
  "dropped": "sandboxDB"
}

schoolDB> use schoolDB
switched to db schoolDB

schoolDB> show dbs
admin      0.000GB
config     0.000GB
local      0.000GB
schoolDB   0.000GB



In [6]:
# 🧪 Automated Verification — Task 1.2
sandbox_gone = "sandboxDB" not in _MONGO_SERVER._databases
curr_is_school = (_MONGO_SERVER.current_db_name == "schoolDB")
record_grade(
    "Task 1.2: Drop Temporary Database",
    bool(sandbox_gone and curr_is_school),
    "sandboxDB was permanently deleted and current context is back on schoolDB!",
    "Ensure sandboxDB was dropped via db.dropDatabase() and you switched back to schoolDB."
)

✅ [PASS] Task 1.2: Drop Temporary Database: sandboxDB was permanently deleted and current context is back on schoolDB!


---
## 🗂️ Lab 2: Collection Operations (`createCollection`, `show collections`, `drop`)

### 📖 Concept Walkthrough
1. **Create Collection (`db.createCollection(name, options)`):**
   - Explicitly creates a new collection in the current database. While MongoDB auto-creates collections on first insert, explicit creation is useful for setting validation rules or capped collection sizes.
2. **List Collections (`show collections`):**
   - Lists every collection that exists in the currently selected database.
3. **Delete Collection (`db.<collection>.drop()`):**
   - Permanently removes a collection, all of its documents, and its indexes.

### 🎯 Hands-On Task 2.1: Explicitly Create & List Collections
**Assigned Task:**
1. Ensure you are in `schoolDB`.
2. Explicitly create a collection named `"courses"` and another temporary collection named `"old_records"` using `db.createCollection()`.
3. Run `show collections` to see all collections in `schoolDB` (`courses`, `old_records`, and `students`).

In [7]:
%%mongosh
// Step 1: Ensure schoolDB is selected
use schoolDB

// Step 2: Explicitly create "courses" and "old_records" collections
db.createCollection("courses");
db.createCollection("old_records");

// Step 3: List all collections in schoolDB
show collections

schoolDB> use schoolDB
switched to db schoolDB

schoolDB> db.createCollection("courses")
{
  "ok": 1
}

schoolDB> db.createCollection("old_records")
{
  "ok": 1
}

schoolDB> show collections
courses
old_records
students



In [8]:
# 🧪 Automated Verification — Task 2.1
school_cols = set(_MONGO_SERVER._databases["schoolDB"]._collections.keys())
record_grade(
    "Task 2.1: Explicit Collection Creation",
    {"students", "courses", "old_records"}.issubset(school_cols),
    f"Collections present in schoolDB: {sorted(school_cols)}",
    "Make sure both 'courses' and 'old_records' were created in schoolDB."
)

✅ [PASS] Task 2.1: Explicit Collection Creation: Collections present in schoolDB: ['courses', 'old_records', 'students']


### 🎯 Hands-On Task 2.2: Drop an Unused Collection (`db.<collection>.drop()`)
**Assigned Task:**
1. Permanently delete both the `old_records` and `courses` collections using `.drop()` so we have a clean workspace focusing on `students`.
2. Run `show collections` to confirm only `students` remains in `schoolDB`.

In [9]:
%%mongosh
// Step 1: Drop the old_records and courses collections
db.old_records.drop();
db.courses.drop();

// Step 2: List remaining collections
show collections

schoolDB> db.old_records.drop()
true

schoolDB> db.courses.drop()
true

schoolDB> show collections
students



In [10]:
# 🧪 Automated Verification — Task 2.2
school_cols = set(_MONGO_SERVER._databases["schoolDB"]._collections.keys())
record_grade(
    "Task 2.2: Drop Collections",
    school_cols == {"students"},
    "old_records and courses were dropped; only 'students' remains in schoolDB!",
    f"Expected only 'students' collection, but found: {sorted(school_cols)}"
)

✅ [PASS] Task 2.2: Drop Collections: old_records and courses were dropped; only 'students' remains in schoolDB!


---
## ✍️ Lab 3: CRUD Operations — Create (`insertOne`, `insertMany`) & Read (`find`, `findOne`)

### 📖 Concept Walkthrough
* **`db.<collection>.insertOne(document)`**: Inserts a single document into a collection (auto-creates the collection if needed).
* **`db.<collection>.insertMany([doc1, doc2, ...])`**: Inserts multiple documents in a single call, passed as an array `[...]`.
* **`db.<collection>.find(query, projection)`**: Retrieves all documents matching the filter and returns a cursor. Calling `find()` with no arguments returns every document.
* **`db.<collection>.findOne(query, projection)`**: Retrieves only the **first** matching document (or `null` if none match) instead of a cursor.

### 🎯 Hands-On Task 3.1: Populate `students` Using `insertOne()` and `insertMany()`
**Assigned Task:**
1. Use `db.students.insertOne()` to insert Rahul's document:
   `{ name: "Rahul", age: 19, grade: 9, subjects: ["Math", "Science"] }`
2. Use `db.students.insertMany()` to insert an array of two documents in a single call:
   - `{ name: "Meera", age: 17, grade: 7, subjects: ["English", "Science"] }`
   - `{ name: "Kabir", age: 18, grade: 8, subjects: ["Math", "History"] }`

In [11]:
%%mongosh
// Step 1: Insert Rahul using insertOne()
db.students.insertOne({
  name: "Rahul",
  age: 19,
  grade: 9,
  subjects: ["Math", "Science"]
});

// Step 2: Insert Meera and Kabir using insertMany()
db.students.insertMany([
  { name: "Meera", age: 17, grade: 7, subjects: ["English", "Science"] },
  { name: "Kabir", age: 18, grade: 8, subjects: ["Math", "History"] }
]);

schoolDB> db.students.insertOne({ name: "Rahul", age: 19, grade: 9, subjects: ["Math", "Science"] })
{
  "acknowledged": true,
  "insertedId": "ObjectId('000000000000000000000002')"
}

schoolDB> db.students.insertMany([ { name: "Meera", age: 17, grade: 7, subjects: ["English", "Science"] }, { name: "Kabir", age: 18, grade: 8, subjects: ["Math", "History"] } ])
{
  "acknowledged": true,
  "insertedIds": {
    "0": "ObjectId('000000000000000000000003')",
    "1": "ObjectId('000000000000000000000004')"
  }
}



In [12]:
# 🧪 Automated Verification — Task 3.1
all_students = db.students.find().to_list()
names = {d["name"] for d in all_students}
rahul = db.students.findOne({"name": "Rahul"})
record_grade(
    "Task 3.1: CRUD Create (insertOne & insertMany)",
    {"Aditi", "Rahul", "Meera", "Kabir"}.issubset(names) and rahul.get("age") == 19,
    f"Successfully inserted documents! Total students in collection: {len(all_students)}",
    "Verify Rahul, Meera, and Kabir were inserted with the exact fields requested."
)

✅ [PASS] Task 3.1: CRUD Create (insertOne & insertMany): Successfully inserted documents! Total students in collection: 4


### 🎯 Hands-On Task 3.2: Query Documents Using `find()` and `findOne()`
**Assigned Task:**
1. Retrieve **all** documents in the `students` collection using `db.students.find()`.
2. Retrieve all students whose `age` is `18` using `db.students.find({ age: 18 })`.
3. Retrieve the single document where `name` is `"Rahul"` using `db.students.findOne({ name: "Rahul" })`.

In [13]:
%%mongosh
// Step 1: Retrieve all students
db.students.find();

// Step 2: Retrieve all students where age equals 18
db.students.find({ age: 18 });

// Step 3: Retrieve the first matching document where name is "Rahul"
db.students.findOne({ name: "Rahul" });

schoolDB> db.students.find()
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')"
  },
  {
    "name": "Rahul",
    "age": 19,
    "grade": 9,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  },
  {
    "name": "Meera",
    "age": 17,
    "grade": 7,
    "subjects": [
      "English",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000003')"
  },
  {
    "name": "Kabir",
    "age": 18,
    "grade": 8,
    "subjects": [
      "Math",
      "History"
    ],
    "_id": "ObjectId('000000000000000000000004')"
  }
]

schoolDB> db.students.find({ age: 18 })
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')"
  },
  {
    "name": "Kabir",
    "age": 18,
    "grade": 8,
    "subjects": [
      "Math",
      "History"
    ],
    "_id": "ObjectId('000000000000000000000004')"
  }
]

schoolDB> db.students.findOne({ na

In [14]:
# 🧪 Automated Verification — Task 3.2
batch = _MONGO_SERVER.last_batch_outputs
passed_3_2 = (
    len(batch) >= 3
    and isinstance(batch[-2], list) and len(batch[-2]) == 2
    and isinstance(batch[-1], dict) and batch[-1].get("name") == "Rahul"
)
record_grade(
    "Task 3.2: CRUD Read (find & findOne)",
    passed_3_2,
    "find() returned the matching list of age-18 students (Aditi & Kabir) and findOne() returned Rahul's document!",
    "Make sure you executed all three queries: find(), find({ age: 18 }), and findOne({ name: 'Rahul' })."
)

✅ [PASS] Task 3.2: CRUD Read (find & findOne): find() returned the matching list of age-18 students (Aditi & Kabir) and findOne() returned Rahul's document!


---
## 🔄 Lab 4: CRUD Operations — Update (`updateOne`, `updateMany`) & Delete (`deleteOne`, `deleteMany`)

### 📖 Concept Walkthrough
* **`db.<collection>.updateOne(filter, update)`**: Updates the **first** document matching the query filter using update operators like `$set` to modify specific fields without replacing the whole document.
* **`db.<collection>.updateMany(filter, update)`**: Applies the update operators to **every** document matching the filter.
* **`db.<collection>.deleteOne(filter)`**: Deletes the **first** document that matches the filter.
* **`db.<collection>.deleteMany(filter)`**: Deletes **every** document matching the filter. Passing an empty filter `{}` removes all documents without dropping the collection itself.

### 🎯 Hands-On Task 4.1: Update Documents with `updateOne()` and `updateMany()`
**Assigned Task:**
1. Update `"Rahul"`'s document using `updateOne()` and `$set` so his `age` becomes `20` and his `grade` becomes `10`.
2. Update **all** students whose `age` is `18` using `updateMany()` and `$set` to add a new field `promoted: true`.
3. Run `db.students.find()` to inspect the updated documents.

In [15]:
%%mongosh
// Step 1: Update Rahul's age to 20 and grade to 10 using updateOne()
db.students.updateOne(
  { name: "Rahul" },
  { $set: { age: 20, grade: 10 } }
);

// Step 2: Mark all students with age 18 as promoted: true using updateMany()
db.students.updateMany(
  { age: 18 },
  { $set: { promoted: true } }
);

// Step 3: Verify the updates
db.students.find();

schoolDB> db.students.updateOne( { name: "Rahul" }, { $set: { age: 20, grade: 10 } } )
{
  "acknowledged": true,
  "matchedCount": 1,
  "modifiedCount": 1
}

schoolDB> db.students.updateMany( { age: 18 }, { $set: { promoted: true } } )
{
  "acknowledged": true,
  "matchedCount": 2,
  "modifiedCount": 2
}

schoolDB> db.students.find()
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Rahul",
    "age": 20,
    "grade": 10,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  },
  {
    "name": "Meera",
    "age": 17,
    "grade": 7,
    "subjects": [
      "English",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000003')"
  },
  {
    "name": "Kabir",
    "age": 18,
    "grade": 8,
    "subjects": [
      "Math",
      "History"
    ],
    "_id": "ObjectId('000000000000000000000004')",
    "promoted": true
  }
]



In [16]:
# 🧪 Automated Verification — Task 4.1
rahul_doc = db.students.findOne({"name": "Rahul"})
age_18_docs = db.students.find({"age": 18}).to_list()
passed_4_1 = (
    rahul_doc and rahul_doc.get("age") == 20 and rahul_doc.get("grade") == 10
    and len(age_18_docs) == 2 and all(d.get("promoted") is True for d in age_18_docs)
)
record_grade(
    "Task 4.1: CRUD Update (updateOne & updateMany)",
    passed_4_1,
    "Rahul's age/grade were updated, and all age-18 students now have promoted: true!",
    "Check that Rahul has age: 20, grade: 10 and age-18 students have promoted: true."
)

✅ [PASS] Task 4.1: CRUD Update (updateOne & updateMany): Rahul's age/grade were updated, and all age-18 students now have promoted: true!


### 🎯 Hands-On Task 4.2: Delete Documents with `deleteOne()` and `deleteMany()`
**Assigned Task:**
1. Delete the document where `name` is `"Kabir"` using `db.students.deleteOne({ name: "Kabir" })`.
2. Delete all documents where `age` is `17` using `db.students.deleteMany({ age: 17 })`.
3. Run `db.students.find()` to verify that only `"Aditi"` and `"Rahul"` remain.

In [17]:
%%mongosh
// Step 1: Delete Kabir using deleteOne()
db.students.deleteOne({ name: "Kabir" });

// Step 2: Delete all students with age 17 using deleteMany()
db.students.deleteMany({ age: 17 });

// Step 3: Check remaining students
db.students.find();

schoolDB> db.students.deleteOne({ name: "Kabir" })
{
  "acknowledged": true,
  "deletedCount": 1
}

schoolDB> db.students.deleteMany({ age: 17 })
{
  "acknowledged": true,
  "deletedCount": 1
}

schoolDB> db.students.find()
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Rahul",
    "age": 20,
    "grade": 10,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  }
]



In [18]:
# 🧪 Automated Verification — Task 4.2
remaining_names = [d["name"] for d in db.students.find().to_list()]
record_grade(
    "Task 4.2: CRUD Delete (deleteOne & deleteMany)",
    remaining_names == ["Aditi", "Rahul"],
    f"Kabir and Meera (age 17) were removed. Remaining students: {remaining_names}",
    f"Expected ['Aditi', 'Rahul'], but found: {remaining_names}"
)

✅ [PASS] Task 4.2: CRUD Delete (deleteOne & deleteMany): Kabir and Meera (age 17) were removed. Remaining students: ['Aditi', 'Rahul']


---
## 🔎 Lab 5: More About `find()` — Filter, Projection, `sort()`, `limit()`, `skip()`, and `findOne()`

### 📖 Concept Walkthrough
1. **Query Filter (`1st argument`):** Narrows results to documents matching field values or operators (`$gte`, `$in`, etc.). An empty filter `{}` matches everything.
2. **Projection (`2nd argument`):** Controls which fields are returned — `1` to include a field, `0` to exclude it (`_id` is included by default unless `_id: 0` is specified).
3. **`sort({ field: 1 | -1 })`:** Orders results (`1` = ascending, `-1` = descending).
4. **`skip(n)` & `limit(n)`:** `skip(n)` omits the first `n` documents; `limit(n)` caps the number of returned documents. Commonly chained together for pagination.
5. **`findOne(query, projection)` Revisited:** Returns a single document (not a cursor) and accepts a projection as its 2nd argument, but does **not** support cursor methods like `.sort()` or `.limit()`.

### 🌱 Seed Additional Student Records for Lab 5
First, let's insert 4 more students so we have 6 students total to practice projections, sorting, and pagination.

In [19]:
%%mongosh
db.students.insertMany([
  { name: "Meera", age: 17, grade: 7, subjects: ["English", "Art"], attempts: 1 },
  { name: "Kabir", age: 18, grade: 8, subjects: ["Math", "Science"], attempts: 2 },
  { name: "Zoya",  age: 19, grade: 9, subjects: ["History", "Math"], attempts: 1 },
  { name: "Dev",   age: 21, grade: 11, subjects: ["Physics", "Math"], attempts: 3 }
]);
db.students.find();

schoolDB> db.students.insertMany([ { name: "Meera", age: 17, grade: 7, subjects: ["English", "Art"], attempts: 1 }, { name: "Kabir", age: 18, grade: 8, subjects: ["Math", "Science"], attempts: 2 }, { name: "Zoya",  age: 19, grade: 9, subjects: ["History", "Math"], attempts: 1 }, { name: "Dev",   age: 21, grade: 11, subjects: ["Physics", "Math"], attempts: 3 } ])
{
  "acknowledged": true,
  "insertedIds": {
    "0": "ObjectId('000000000000000000000005')",
    "1": "ObjectId('000000000000000000000006')",
    "2": "ObjectId('000000000000000000000007')",
    "3": "ObjectId('000000000000000000000008')"
  }
}

schoolDB> db.students.find()
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Rahul",
    "age": 20,
    "grade": 10,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  },
  {
    "name": "Meera",
    "age": 17,
    "grade": 7,
 

### 🎯 Hands-On Task 5.1: Filter, Projection & `findOne()` Revisited
**Assigned Task:**
1. Find all students with `age` greater than or equal to `18` (`{ age: { $gte: 18 } }`).
2. Find **all** students (`{}`), returning **only** their `name` and `age` fields while excluding `_id` (`{ name: 1, age: 1, _id: 0 }`).
3. Use `findOne()` to return **only** the `name` field (excluding `_id`) of the first student whose `age` is `>= 18`.

In [20]:
%%mongosh
// Step 1: Find students with age >= 18
db.students.find({ age: { $gte: 18 } });

// Step 2: Return only name and age (exclude _id) for every student
db.students.find({}, { name: 1, age: 1, _id: 0 });

// Step 3: Use findOne() with filter (age >= 18) and projection (name only, exclude _id)
db.students.findOne(
  { age: { $gte: 18 } },
  { name: 1, _id: 0 }
);

schoolDB> db.students.find({ age: { $gte: 18 } })
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Rahul",
    "age": 20,
    "grade": 10,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  },
  {
    "name": "Kabir",
    "age": 18,
    "grade": 8,
    "subjects": [
      "Math",
      "Science"
    ],
    "attempts": 2,
    "_id": "ObjectId('000000000000000000000006')"
  },
  {
    "name": "Zoya",
    "age": 19,
    "grade": 9,
    "subjects": [
      "History",
      "Math"
    ],
    "attempts": 1,
    "_id": "ObjectId('000000000000000000000007')"
  },
  {
    "name": "Dev",
    "age": 21,
    "grade": 11,
    "subjects": [
      "Physics",
      "Math"
    ],
    "attempts": 3,
    "_id": "ObjectId('000000000000000000000008')"
  }
]

schoolDB> db.students.find({}, { name: 1, age: 1, _id: 0 })
[
  {
    "name": "Aditi",
    "a

In [21]:
# 🧪 Automated Verification — Task 5.1
outs = _MONGO_SERVER.last_batch_outputs
passed_5_1 = (
    len(outs) >= 3
    and isinstance(outs[-2], list) and len(outs[-2]) == 6
    and set(outs[-2][0].keys()) == {"name", "age"}
    and outs[-1] == {"name": "Aditi"}
)
record_grade(
    "Task 5.1: Filter, Projection & findOne()",
    passed_5_1,
    "Projection cleanly excluded _id and returned exact requested fields!",
    "Verify your projection arguments use { name: 1, age: 1, _id: 0 } and { name: 1, _id: 0 }."
)

✅ [PASS] Task 5.1: Filter, Projection & findOne(): Projection cleanly excluded _id and returned exact requested fields!


### 🎯 Hands-On Task 5.2: Sorting & Pagination (`sort()`, `skip()`, `limit()`)
**Assigned Task:**
1. Retrieve all students sorted by `age` from **highest to lowest** (`sort({ age: -1 })`).
2. Paginate the student list: sort students by `age` in **ascending** order (`1`), **skip** the first `2` documents (`skip(2)`), and **limit** the result to `3` documents (`limit(3)`).

In [22]:
%%mongosh
// Step 1: Sort students from highest to lowest age (-1)
db.students.find().sort({ age: -1 });

// Step 2: Sort by age ascending (1), skip first 2 results, and return the next 3
db.students.find().sort({ age: 1 }).skip(2).limit(3);

schoolDB> db.students.find().sort({ age: -1 })
[
  {
    "name": "Dev",
    "age": 21,
    "grade": 11,
    "subjects": [
      "Physics",
      "Math"
    ],
    "attempts": 3,
    "_id": "ObjectId('000000000000000000000008')"
  },
  {
    "name": "Rahul",
    "age": 20,
    "grade": 10,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  },
  {
    "name": "Zoya",
    "age": 19,
    "grade": 9,
    "subjects": [
      "History",
      "Math"
    ],
    "attempts": 1,
    "_id": "ObjectId('000000000000000000000007')"
  },
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Kabir",
    "age": 18,
    "grade": 8,
    "subjects": [
      "Math",
      "Science"
    ],
    "attempts": 2,
    "_id": "ObjectId('000000000000000000000006')"
  },
  {
    "name": "Meera",
    "age": 17,
    "grade": 7,
    "subjects": [
      "English",
      "

In [23]:
# 🧪 Automated Verification — Task 5.2
outs = _MONGO_SERVER.last_batch_outputs
desc_ages = [d["age"] for d in outs[-2]] if len(outs) >= 2 and isinstance(outs[-2], list) else []
page_ages = [d["age"] for d in outs[-1]] if len(outs) >= 1 and isinstance(outs[-1], list) else []
record_grade(
    "Task 5.2: Cursor Sort, Skip & Limit",
    desc_ages == sorted(desc_ages, reverse=True) and len(page_ages) == 3 and page_ages == [18, 19, 20],
    f"Descending sort ages: {desc_ages} | Paginated slice (skip 2, limit 3) ages: {page_ages}",
    "Check your .sort({ age: -1 }) and .sort({ age: 1 }).skip(2).limit(3) commands."
)

✅ [PASS] Task 5.2: Cursor Sort, Skip & Limit: Descending sort ages: [21, 20, 19, 18, 18, 17] | Paginated slice (skip 2, limit 3) ages: [18, 19, 20]


---
## ⚡ Lab 6: Query & Update Operators in Action

### 📖 Concept Walkthrough

#### 1. Comparison, Array & Logical Query Operators
| Category | Operator | Description | Example |
| :--- | :--- | :--- | :--- |
| **Comparison** | `$gt` / `$gte` | Greater than / Greater than or equal | `{ age: { $gt: 17 } }` |
| **Comparison** | `$lt` / `$lte` | Less than / Less than or equal | `{ age: { $lte: 20 } }` |
| **Comparison** | `$ne` | Not equal | `{ age: { $ne: 18 } }` |
| **Array / Set** | `$in` / `$nin` | Matches any value in array / Not in array | `{ age: { $in: [17, 18, 19] } }` |
| **Logical** | `$or` / `$and` | Joins condition objects with OR / AND | `{ $or: [{ age: { $gte: 19 } }, { name: "Rahul" }] }` |

#### 2. Update Operators
| Operator | Description | Example |
| :--- | :--- | :--- |
| **`$set`** | Sets or replaces a field's value | `{ $set: { age: 20 } }` |
| **`$inc`** | Increments a numeric field by a specified amount | `{ $inc: { attempts: 1 } }` |
| **`$push`** | Appends an element to an array field | `{ $push: { subjects: "History" } }` |
| **`$pull`** | Removes a matching element from an array field | `{ $pull: { subjects: "Science" } }` |

---
### 🎯 Hands-On Task 6.1: Comparison, Array & Logical Query Operators
**Assigned Task:**
1. Find students whose `age` is strictly greater than `17` **and** strictly less than `21` (`{ age: { $gt: 17, $lt: 21 } }`).
2. Find students whose `age` is in `[17, 18, 19]` using `$in`.
3. Find students where `age >= 20` **OR** `name` is `"Meera"` using `$or`.

In [24]:
%%mongosh
// Step 1: Find students with age strictly between 17 and 21 ($gt: 17, $lt: 21)
db.students.find({ age: { $gt: 17, $lt: 21 } });

// Step 2: Find students where age is in [17, 18, 19] using $in
db.students.find({ age: { $in: [17, 18, 19] } });

// Step 3: Find students where age >= 20 OR name is "Meera" using $or
db.students.find({
  $or: [{ age: { $gte: 20 } }, { name: "Meera" }]
});

schoolDB> db.students.find({ age: { $gt: 17, $lt: 21 } })
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Rahul",
    "age": 20,
    "grade": 10,
    "subjects": [
      "Math",
      "Science"
    ],
    "_id": "ObjectId('000000000000000000000002')"
  },
  {
    "name": "Kabir",
    "age": 18,
    "grade": 8,
    "subjects": [
      "Math",
      "Science"
    ],
    "attempts": 2,
    "_id": "ObjectId('000000000000000000000006')"
  },
  {
    "name": "Zoya",
    "age": 19,
    "grade": 9,
    "subjects": [
      "History",
      "Math"
    ],
    "attempts": 1,
    "_id": "ObjectId('000000000000000000000007')"
  }
]

schoolDB> db.students.find({ age: { $in: [17, 18, 19] } })
[
  {
    "name": "Aditi",
    "age": 18,
    "grade": 8,
    "_id": "ObjectId('000000000000000000000001')",
    "promoted": true
  },
  {
    "name": "Meera",
    "age": 17,
    "grade": 7,
    "subjects": [
 

In [25]:
# 🧪 Automated Verification — Task 6.1
outs = _MONGO_SERVER.last_batch_outputs
q1_names = {d["name"] for d in outs[-3]} if len(outs) >= 3 else set()
q2_names = {d["name"] for d in outs[-2]} if len(outs) >= 2 else set()
q3_names = {d["name"] for d in outs[-1]} if len(outs) >= 1 else set()
passed_6_1 = (
    q1_names == {"Aditi", "Rahul", "Kabir", "Zoya"}
    and q2_names == {"Aditi", "Meera", "Kabir", "Zoya"}
    and q3_names == {"Rahul", "Meera", "Dev"}
)
record_grade(
    "Task 6.1: Query Operators ($gt, $lt, $in, $or)",
    passed_6_1,
    "All comparison, array ($in), and logical ($or) operator queries returned exact matches!",
    "Double-check the filter conditions for $gt/$lt, $in, and $or."
)

✅ [PASS] Task 6.1: Query Operators ($gt, $lt, $in, $or): All comparison, array ($in), and logical ($or) operator queries returned exact matches!


### 🎯 Hands-On Task 6.2: Update Operators (`$set`, `$inc`, `$push`, `$pull`)
**Assigned Task:**
1. In a single `db.students.updateOne()` call for `{ name: "Rahul" }`:
   - Set `age` to `21` using `$set`
   - Increment `attempts` by `1` using `$inc`
   - Add `"History"` to the `subjects` array using `$push`
2. In a second `db.students.updateOne()` call for `{ name: "Rahul" }`:
   - Remove `"Science"` from the `subjects` array using `$pull`
3. Retrieve `"Rahul"`'s updated document with `db.students.findOne({ name: "Rahul" })`.

In [26]:
%%mongosh
// Step 1: Update Rahul using $set, $inc, and $push together
db.students.updateOne(
  { name: "Rahul" },
  {
    $set: { age: 21 },
    $inc: { attempts: 1 },
    $push: { subjects: "History" }
  }
);

// Step 2: Remove "Science" from Rahul's subjects array using $pull
db.students.updateOne(
  { name: "Rahul" },
  { $pull: { subjects: "Science" } }
);

// Step 3: Inspect Rahul's final document
db.students.findOne({ name: "Rahul" });

schoolDB> db.students.updateOne( { name: "Rahul" }, { $set: { age: 21 }, $inc: { attempts: 1 }, $push: { subjects: "History" } } )
{
  "acknowledged": true,
  "matchedCount": 1,
  "modifiedCount": 1
}

schoolDB> db.students.updateOne( { name: "Rahul" }, { $pull: { subjects: "Science" } } )
{
  "acknowledged": true,
  "matchedCount": 1,
  "modifiedCount": 1
}

schoolDB> db.students.findOne({ name: "Rahul" })
{
  "name": "Rahul",
  "age": 21,
  "grade": 10,
  "subjects": [
    "Math",
    "History"
  ],
  "_id": "ObjectId('000000000000000000000002')",
  "attempts": 1
}



In [27]:
# 🧪 Automated Verification — Task 6.2
rahul_now = db.students.findOne({"name": "Rahul"})
passed_6_2 = (
    rahul_now is not None
    and rahul_now.get("age") == 21
    and rahul_now.get("attempts") == 1
    and rahul_now.get("subjects") == ["Math", "History"]
)
record_grade(
    "Task 6.2: Update Operators ($set, $inc, $push, $pull)",
    passed_6_2,
    f"Rahul's updated profile -> age: {rahul_now.get('age')}, attempts: {rahul_now.get('attempts')}, subjects: {rahul_now.get('subjects')}",
    "Ensure $set, $inc, $push, and $pull were all applied to Rahul's document."
)

✅ [PASS] Task 6.2: Update Operators ($set, $inc, $push, $pull): Rahul's updated profile -> age: 21, attempts: 1, subjects: ['Math', 'History']


---
## 🏆 Lab 7: Capstone Real-World Lab Challenge

Put everything you have learned together in one end-to-end workflow!

### 🎯 Hands-On Task 7.1: University Library Database Workflow
**Assigned Task:**
1. Switch to a new database named `libraryDB` (`use libraryDB`).
2. Explicitly create a collection named `"books"` (`db.createCollection("books")`).
3. Use `insertMany()` to insert the following 4 book documents into `db.books`:
   - `{ title: "MongoDB Basics", copies: 4, tags: ["NoSQL", "Database"], checkouts: 10 }`
   - `{ title: "System Design", copies: 2, tags: ["Architecture", "Backend"], checkouts: 25 }`
   - `{ title: "Data Structures", copies: 6, tags: ["CS", "Algorithms"], checkouts: 40 }`
   - `{ title: "Legacy SQL", copies: 0, tags: ["SQL", "Archive"], checkouts: 5 }`
4. Use `updateOne()` on `{ title: "MongoDB Basics" }` to:
   - Increment `checkouts` by `1` (`$inc`)
   - Push `"Bestseller"` into `tags` (`$push`)
5. Use `deleteOne()` to delete books where `copies` equals `0` (`{ copies: 0 }`).
6. Query `db.books.find()` for books with `copies >= 2`, projecting only `{ title: 1, checkouts: 1, _id: 0 }`, sorted by `checkouts` **descending** (`-1`), limited to the top `2` books.

In [28]:
%%mongosh
// 1. Switch to libraryDB and explicitly create the "books" collection
use libraryDB
db.createCollection("books");

// 2. Insert the 4 books using insertMany()
db.books.insertMany([
  { title: "MongoDB Basics", copies: 4, tags: ["NoSQL", "Database"], checkouts: 10 },
  { title: "System Design", copies: 2, tags: ["Architecture", "Backend"], checkouts: 25 },
  { title: "Data Structures", copies: 6, tags: ["CS", "Algorithms"], checkouts: 40 },
  { title: "Legacy SQL", copies: 0, tags: ["SQL", "Archive"], checkouts: 5 }
]);

// 3. Increment checkouts by 1 and push "Bestseller" tag for "MongoDB Basics"
db.books.updateOne(
  { title: "MongoDB Basics" },
  {
    $inc: { checkouts: 1 },
    $push: { tags: "Bestseller" }
  }
);

// 4. Delete out-of-stock book where copies == 0
db.books.deleteOne({ copies: 0 });

// 5. Return top 2 books with copies >= 2, projecting title & checkouts (no _id), sorted by checkouts descending
db.books.find(
  { copies: { $gte: 2 } },
  { title: 1, checkouts: 1, _id: 0 }
).sort({ checkouts: -1 }).limit(2);

libraryDB> use libraryDB
switched to db libraryDB

libraryDB> db.createCollection("books")
{
  "ok": 1
}

libraryDB> db.books.insertMany([ { title: "MongoDB Basics", copies: 4, tags: ["NoSQL", "Database"], checkouts: 10 }, { title: "System Design", copies: 2, tags: ["Architecture", "Backend"], checkouts: 25 }, { title: "Data Structures", copies: 6, tags: ["CS", "Algorithms"], checkouts: 40 }, { title: "Legacy SQL", copies: 0, tags: ["SQL", "Archive"], checkouts: 5 } ])
{
  "acknowledged": true,
  "insertedIds": {
    "0": "ObjectId('000000000000000000000001')",
    "1": "ObjectId('000000000000000000000002')",
    "2": "ObjectId('000000000000000000000003')",
    "3": "ObjectId('000000000000000000000004')"
  }
}

libraryDB> db.books.updateOne( { title: "MongoDB Basics" }, { $inc: { checkouts: 1 }, $push: { tags: "Bestseller" } } )
{
  "acknowledged": true,
  "matchedCount": 1,
  "modifiedCount": 1
}

libraryDB> db.books.deleteOne({ copies: 0 })
{
  "acknowledged": true,
  "deletedCount":

In [29]:
# 🧪 Automated Verification — Task 7.1 (Capstone)
lib_db = _MONGO_SERVER._databases.get("libraryDB")
top_2 = _MONGO_SERVER.last_output
mb_book = lib_db.books.findOne({"title": "MongoDB Basics"}) if lib_db else None
total_books = len(lib_db.books.find().to_list()) if lib_db else 0

passed_7_1 = (
    lib_db is not None
    and total_books == 3
    and mb_book and mb_book.get("checkouts") == 11 and "Bestseller" in mb_book.get("tags", [])
    and top_2 == [
        {"title": "Data Structures", "checkouts": 40},
        {"title": "System Design", "checkouts": 25}
    ]
)
record_grade(
    "Task 7.1: Capstone Library Workflow",
    passed_7_1,
    "Capstone completed! libraryDB created, books inserted, updated, cleaned, and top 2 queried!",
    "Verify all 5 steps in Task 7.1 were executed accurately."
)

✅ [PASS] Task 7.1: Capstone Library Workflow: Capstone completed! libraryDB created, books inserted, updated, cleaned, and top 2 queried!


---
## 📊 Final Lab Grade Report
Run the cell below to view your final completion status across all hands-on lab tasks!

In [30]:
# 🏆 Display Final Scoreboard
show_lab_report()

🏆 MONGODB HANDS-ON LAB SCOREBOARD: 13/13 Tasks Completed
  ✅ PASS  |  Task 1.1: Create & Use schoolDB
  ✅ PASS  |  Task 1.2: Drop Temporary Database
  ✅ PASS  |  Task 2.1: Explicit Collection Creation
  ✅ PASS  |  Task 2.2: Drop Collections
  ✅ PASS  |  Task 3.1: CRUD Create (insertOne & insertMany)
  ✅ PASS  |  Task 3.2: CRUD Read (find & findOne)
  ✅ PASS  |  Task 4.1: CRUD Update (updateOne & updateMany)
  ✅ PASS  |  Task 4.2: CRUD Delete (deleteOne & deleteMany)
  ✅ PASS  |  Task 5.1: Filter, Projection & findOne()
  ✅ PASS  |  Task 5.2: Cursor Sort, Skip & Limit
  ✅ PASS  |  Task 6.1: Query Operators ($gt, $lt, $in, $or)
  ✅ PASS  |  Task 6.2: Update Operators ($set, $inc, $push, $pull)
  ✅ PASS  |  Task 7.1: Capstone Library Workflow
